[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/34_representation_learning/01_autoencoders.ipynb)

# Bài 01 — Autoencoder

**🎯 Mục tiêu**
- Ý tưởng **học biểu diễn không giám sát**: nén dữ liệu qua một nút cổ chai rồi tái tạo lại.
- Autoencoder **tuyến tính** ≈ PCA (kiểm chứng bằng số); autoencoder tích chập vượt PCA cùng số chiều.
- Biểu diễn học được **chứa gì**? Đo tương quan giữa latent với vị trí, kích thước, loại hình.
- Ứng dụng: **khử nhiễu** (denoising AE), **phát hiện bất thường** bằng sai số tái tạo — và giới hạn của chúng.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/34_representation_learning

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import roc_auc_score

from ml_course.datasets import SHAPE_CLASSES, SHAPE_CLASSES_EXTRA, make_shapes
from ml_course.visualization import set_style

try:
    import torch
    import torch.nn.functional as F
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
print("thiết bị:", device)

X_train, y_train = make_shapes(20000, seed=0, noise=0.05)
X_test, y_test = make_shapes(2000, seed=1, noise=0.05)
to_tensor = lambda X: torch.tensor(X[:, None], dtype=torch.float32)
X_train_t, X_test_t = to_tensor(X_train), to_tensor(X_test)
X_flat, X_test_flat = X_train.reshape(len(X_train), -1), X_test.reshape(len(X_test), -1)
print(f"train {X_train.shape} | test {X_test.shape} — nhãn chỉ dùng để ĐÁNH GIÁ, không dùng khi học biểu diễn")

## 1. Ý tưởng

**Autoencoder**: encoder $f$ nén ảnh $x$ (784 chiều) thành vector **latent** $z = f(x)$ ít chiều; decoder $g$ tái tạo $\hat x = g(z)$; huấn luyện cực tiểu sai số tái tạo $\|x - g(f(x))\|^2$. Không cần nhãn — dữ liệu tự làm "đáp án" (**self-supervised**). Nút cổ chai buộc $z$ giữ lại những gì **cần nhất** để tái tạo.

### Autoencoder tuyến tính ≈ PCA

Nếu encoder/decoder đều tuyến tính và loss là MSE, nghiệm tối ưu của autoencoder **trải cùng không gian con** với $k$ thành phần chính của PCA (chương 21) — Baldi & Hornik (1989). Kiểm chứng: so sai số tái tạo và các **góc chính** (principal angles) giữa hai không gian con 16 chiều.

In [ ]:
k = 16
pca16 = PCA(k).fit(X_flat)
mse_pca = ((pca16.inverse_transform(pca16.transform(X_test_flat)) - X_test_flat) ** 2).mean()

mean = torch.tensor(X_flat.mean(0), dtype=torch.float32)
Xc, Xc_test = torch.tensor(X_flat) - mean, torch.tensor(X_test_flat) - mean
torch.manual_seed(0)
linear_ae = nn.Sequential(nn.Linear(784, k, bias=False), nn.Linear(k, 784, bias=False)).to(device)
opt = torch.optim.Adam(linear_ae.parameters(), lr=1e-3)
g = torch.Generator().manual_seed(0)
for step in range(6000):
    xb = Xc[torch.randint(0, len(Xc), (256,), generator=g)].to(device)
    opt.zero_grad()
    F.mse_loss(linear_ae(xb), xb).backward()
    opt.step()
with torch.no_grad():
    mse_lin = F.mse_loss(linear_ae(Xc_test.to(device)), Xc_test.to(device)).item()
Q, _ = np.linalg.qr(linear_ae[1].weight.detach().cpu().numpy())        # cơ sở trực chuẩn của không gian con decoder
cosines = np.linalg.svd(Q.T @ pca16.components_.T, compute_uv=False)
print(f"MSE tái tạo: PCA = {mse_pca:.5f} | AE tuyến tính = {mse_lin:.5f}")
ev = PCA(20).fit(X_flat).explained_variance_
print(f"cos các góc chính giữa hai không gian con (1 = trùng nhau): {np.round(cosines, 2)}")
print(f"phương sai của thành phần chính 13–20: {np.round(ev[12:20], 2)}")

Sai số tái tạo gần như **bằng nhau**, và gần như mọi góc chính có cos ≈ 1 — hai không gian con gần trùng. Chỉ hướng cuối cùng lệch: các thành phần chính quanh vị trí thứ 16 có phương sai **xấp xỉ bằng nhau** (dòng in thứ hai), nên đổi hướng trong vùng đó gần như không làm thay đổi sai số — gradient descent không có lý do để chọn đúng các trục PCA (và chưa hội tụ hết ở các hướng phương sai nhỏ). Khác biệt cốt lõi: PCA cho các trục **trực giao, xếp theo phương sai**; AE tuyến tính cho một cơ sở bất kỳ của (gần) cùng không gian con. Sức mạnh thật của autoencoder đến từ **phi tuyến**.

## 2. Autoencoder tích chập

Encoder: 2 conv stride 2 (28 → 14 → 7) → tuyến tính → $z \in \mathbb R^k$. Decoder đối xứng với **ConvTranspose** (phóng to) và sigmoid ở đầu ra (pixel trong [0, 1]).

In [ ]:
class ConvAE(nn.Module):
    def __init__(self, k):
        super().__init__()
        self.encoder = nn.Sequential(nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
                                     nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(), nn.Flatten(), nn.Linear(64 * 7 * 7, k))
        self.decoder = nn.Sequential(nn.Linear(k, 64 * 7 * 7), nn.ReLU(), nn.Unflatten(1, (64, 7, 7)),
                                     nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(),
                                     nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1), nn.Sigmoid())

    def forward(self, x):
        return self.decoder(self.encoder(x))


def train_ae(k, steps=3000, input_noise=0.0, seed=0):
    torch.manual_seed(seed)
    model = ConvAE(k).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        xb = X_train_t[torch.randint(0, len(X_train_t), (128,), generator=g)]
        x_in = (xb + input_noise * torch.randn(xb.shape, generator=g)).clamp(0, 1) if input_noise else xb
        opt.zero_grad()
        F.mse_loss(model(x_in.to(device)), xb.to(device)).backward()           # đích luôn là ảnh SẠCH
        opt.step()
    return model.eval()


aes = {}
for k in [8, 32]:
    t0 = time.perf_counter()
    aes[k] = train_ae(k)
    with torch.no_grad():
        mse_ae = F.mse_loss(aes[k](X_test_t.to(device)), X_test_t.to(device)).item()
    p = PCA(k).fit(X_flat)
    mse_p = ((p.inverse_transform(p.transform(X_test_flat)) - X_test_flat) ** 2).mean()
    print(f"k = {k:2d}: MSE autoencoder = {mse_ae:.5f} | MSE PCA = {mse_p:.5f} | {time.perf_counter() - t0:.0f}s")

In [ ]:
with torch.no_grad():
    rec = {k: m(X_test_t[:8].to(device)).cpu() for k, m in aes.items()}
p8 = PCA(8).fit(X_flat)
rec_pca8 = p8.inverse_transform(p8.transform(X_test_flat[:8])).reshape(-1, 28, 28)
rows = [("gốc", X_test[:8]), ("PCA k=8", rec_pca8), ("AE k=8", rec[8][:, 0]), ("AE k=32", rec[32][:, 0])]
fig, axes = plt.subplots(4, 8, figsize=(12, 6.2))
for r, (name, imgs) in enumerate(rows):
    for j in range(8):
        axes[r, j].imshow(imgs[j], cmap="gray", vmin=0, vmax=1)
        axes[r, j].axis("off")
    axes[r, 0].set_title(name, fontsize=9, loc="left")
plt.tight_layout()
plt.show()

Cùng 8 chiều, PCA chỉ tái tạo được "đốm mờ" đúng vị trí (tổ hợp tuyến tính của các ảnh cơ sở); autoencoder phi tuyến tái tạo được hình dạng rõ hơn hẳn — sai số thấp hơn khoảng một nửa.

## 3. Latent chứa gì?

Không dùng nhãn khi học, vậy $z$ mã hoá những thuộc tính nào? Đo các thuộc tính "thật" của mỗi ảnh (tâm $x$, tâm $y$, diện tích của hình — tính từ ảnh sạch) và loại hình, rồi xem chúng **dự đoán được** từ $z$ tốt đến đâu (hồi quy/phân loại tuyến tính trên $z$).

In [ ]:
mask = X_test > 0.4
yy, xx = np.mgrid[0:28, 0:28]
area = mask.sum((1, 2))
props = {"tâm x": (mask * xx).sum((1, 2)) / area, "tâm y": (mask * yy).sum((1, 2)) / area, "diện tích": area}
with torch.no_grad():
    Z = aes[32].encoder(X_test_t.to(device)).cpu().numpy()
half = len(Z) // 2
for name, v in props.items():
    r2 = LinearRegression().fit(Z[:half], v[:half]).score(Z[half:], v[half:])
    print(f"{name:10s}: R² hồi quy tuyến tính từ latent = {r2:.3f}")
probe = LogisticRegression(max_iter=5000).fit(Z[:half], y_test[:half])
print(f"loại hình  : accuracy phân loại tuyến tính từ latent = {probe.score(Z[half:], y_test[half:]):.3f} (đoán ngẫu nhiên: 0.2)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
Z2 = PCA(2).fit_transform(Z)
sc = axes[0].scatter(Z2[:, 0], Z2[:, 1], c=props["tâm x"], s=6, cmap="viridis")
plt.colorbar(sc, ax=axes[0], label="tâm x của hình")
axes[0].set_title("latent (PCA 2D) tô màu theo VỊ TRÍ")
for c in range(5):
    axes[1].scatter(Z2[y_test == c, 0], Z2[y_test == c, 1], s=6, label=SHAPE_CLASSES[c], alpha=0.6)
axes[1].legend(fontsize=8, markerscale=2)
axes[1].set_title("cùng latent tô màu theo LOẠI HÌNH")
plt.tight_layout()
plt.show()

Latent mã hoá **vị trí và kích thước** gần như tuyến tính (R² cao), nhưng **loại hình** thì tách rất kém. Lý do: sai số tái tạo theo pixel bị chi phối bởi việc đặt đúng chỗ một đốm sáng đúng cỡ; sự khác biệt tròn/vuông/tam giác chỉ ảnh hưởng vài pixel ở biên. Mục tiêu tái tạo **không** nói gì về điều ta quan tâm — đây là động lực cho học tương phản (bài 03), nơi ta **chỉ định** những gì biểu diễn cần bất biến.

## 4. Denoising autoencoder

Huấn luyện với đầu vào **bị nhiễu** nhưng đích là ảnh **sạch** (Vincent et al., 2008): mô hình phải học cấu trúc của dữ liệu để loại nhiễu, thay vì chỉ sao chép.

In [ ]:
dae = train_ae(32, input_noise=0.3)
noisy = (X_test_t + 0.3 * torch.randn(X_test_t.shape, generator=torch.Generator().manual_seed(1))).clamp(0, 1)
with torch.no_grad():
    out_ae, out_dae = aes[32](noisy.to(device)).cpu(), dae(noisy.to(device)).cpu()
print(f"MSE so với ảnh sạch: ảnh nhiễu = {F.mse_loss(noisy, X_test_t):.4f} | AE thường = {F.mse_loss(out_ae, X_test_t):.4f} | "
      f"denoising AE = {F.mse_loss(out_dae, X_test_t):.4f}")
fig, axes = plt.subplots(4, 8, figsize=(12, 6.2))
for r, (name, imgs) in enumerate([("sạch", X_test_t), ("nhiễu σ=0.3", noisy), ("AE thường", out_ae), ("denoising AE", out_dae)]):
    for j in range(8):
        axes[r, j].imshow(imgs[j, 0], cmap="gray", vmin=0, vmax=1)
        axes[r, j].axis("off")
    axes[r, 0].set_title(name, fontsize=9, loc="left")
plt.tight_layout()
plt.show()

AE thường chưa từng thấy nhiễu nên tái tạo lẫn cả nhiễu (còn tệ hơn ảnh nhiễu); denoising AE giảm sai số nhiều lần. Ý tưởng "học cách khử nhiễu" chính là nền tảng của **mô hình khuếch tán** (diffusion models) — sinh ảnh bằng cách khử nhiễu dần từ nhiễu thuần (bài 02).

## 5. Phát hiện bất thường bằng sai số tái tạo

Ý tưởng: AE chỉ học dữ liệu **bình thường** → tái tạo tốt cái quen, kém cái lạ → sai số tái tạo cao = bất thường. Thử: tập kiểm tra gồm 1 000 ảnh 5 hình đã biết + 1 000 ảnh 4 hình **mới** (ngôi sao, lục giác, bán nguyệt, chữ L).

In [ ]:
X_new, _ = make_shapes(1000, seed=3, noise=0.05, classes=SHAPE_CLASSES_EXTRA)
X_mix = torch.cat([X_test_t[:1000], to_tensor(X_new)])
is_anomaly = np.r_[np.zeros(1000), np.ones(1000)]
for k, m in aes.items():
    with torch.no_grad():
        err = ((m(X_mix.to(device)).cpu() - X_mix) ** 2).mean((1, 2, 3)).numpy()
    print(f"AE k = {k:2d}: ROC-AUC phát hiện hình mới = {roc_auc_score(is_anomaly, err):.3f}")
p32 = PCA(32).fit(X_flat)
flat_mix = X_mix.numpy().reshape(len(X_mix), -1)
err_pca = ((p32.inverse_transform(p32.transform(flat_mix)) - flat_mix) ** 2).mean(1)
print(f"PCA k = 32: ROC-AUC = {roc_auc_score(is_anomaly, err_pca):.3f}")

Kết quả chỉ ở mức **trung bình**: autoencoder tổng quát hoá "quá tốt" — nó học cách tái tạo các đốm sáng ở mọi vị trí, nên tái tạo được cả hình mới với sai số không khác mấy (và PCA còn tệ hơn đoán ngẫu nhiên: các hình mới tình cờ dễ tái tạo tuyến tính hơn). Sai số tái tạo chỉ phát hiện được bất thường khi chúng **khác biệt ở chính những gì AE mã hoá** (vị trí, độ sáng, kết cấu khác hẳn). Trong thực tế (chương 22): so với các phương pháp khác, dùng ngưỡng theo validation, và kiểm tra trên **bất thường thật** chứ không chỉ bất thường tổng hợp.

## 🧠 Kiểm tra nhanh

1. Autoencoder học gì khi không có nhãn? Vai trò của nút cổ chai?
2. Quan hệ giữa autoencoder tuyến tính và PCA?
3. Vì sao latent của AE mã hoá vị trí tốt hơn loại hình?
4. Denoising AE khác AE thường thế nào? Liên hệ với diffusion models?
5. Khi nào phát hiện bất thường bằng sai số tái tạo hoạt động kém?

## 📌 Tóm tắt

- AE = encoder → latent ít chiều → decoder, học bằng tái tạo; AE tuyến tính ≈ PCA, AE phi tuyến nén tốt hơn nhiều.
- Biểu diễn phản ánh **mục tiêu huấn luyện**: tái tạo pixel ưu tiên vị trí/kích thước, không nhất thiết là thông tin ta cần.
- Denoising AE học cấu trúc dữ liệu (nền tảng của diffusion); phát hiện bất thường bằng sai số tái tạo có giới hạn khi AE tổng quát hoá quá tốt.

## 📚 Tài liệu tham khảo

- Hinton & Salakhutdinov (2006) — *Reducing the Dimensionality of Data with Neural Networks*
- Baldi & Hornik (1989) — *Neural networks and principal component analysis*
- Vincent et al. (2008) — *Extracting and Composing Robust Features with Denoising Autoencoders*
- Goodfellow, Bengio & Courville — *Deep Learning*, chương 14 (Autoencoders)
- Bengio, Courville & Vincent (2013) — *Representation Learning: A Review and New Perspectives*